# Topic 5 — Ứng dụng AI phát hiện và phân vùng lỗi bề mặt sản phẩm công nghiệp từ hình ảnh

**Nhóm:** Nhóm 7 · **Hướng nghiên cứu:** AI dự đoán · **Người dùng chính:** nhân viên QC  
**Dữ liệu:** NEU-DET (Bounding Box) và NEU-Seg ( Mask)  
**Mô hình chính:** YOLOv8s (Object Detection) và U-Net (Semantic Segmentation), xử lý độc lập  
**Tầng ứng dụng:** Captioning/VQA tiếng Việt từ JSON kết quả, dùng LLM/prompt có sẵn và không fine-tune  
**Phạm vi:** ảnh tĩnh; video, camera giám sát và thời gian thực nằm ngoài phạm vi

> **Quy ước:** hai mô hình chạy độc lập trên cùng một ảnh và cùng hiển thị kết quả; việc này **không** phải Model Fusion (chỉ là hướng mở rộng ở mục 2.12). Tầng Captioning/VQA chỉ diễn giải JSON, không tạo box/mask mới và không thay đổi dự đoán.

**Mục lục:** 
**1. Define:** 1.1 Bối cảnh và bài toán · 1.2 Người dùng · 1.3 Dữ liệu · 1.4 Phương pháp · 1.5 Phạm vi · 1.6 Sơ đồ tổng quan · 1.7 Sản phẩm và đánh giá  
**2. Design:** 2.1 Nguyên tắc · 2.2 Kiến trúc · 2.3 Dữ liệu · 2.4 Mô hình chính · **2.5 Baseline** · 2.6 Suy luận độc lập · 2.7 Huấn luyện · 2.8 Đánh giá · 2.9 Demo (Captioning/VQA) · 2.10 Kế hoạch 8 tuần · 2.11 Quyết định và rủi ro · 2.12 Mở rộng

<a id="define"></a>

# 1. Define — Xác định bài toán

<a id="define-1"></a>

## 1.1 Bối cảnh và bài toán

Kiểm tra bề mặt là công đoạn phát hiện khuyết điểm trước khi sản phẩm sang công đoạn sau. Hiện việc này dựa nhiều vào quan sát thủ công, tốn thời gian và phụ thuộc kinh nghiệm người kiểm tra. Đề tài xây công cụ hỗ trợ nhân viên QC đọc **ảnh tĩnh** của sản phẩm, chỉ ra loại lỗi, vị trí và phạm vi lỗi. Kết quả chỉ mang tính hỗ trợ; QC quyết định cuối cùng.

Hai bài toán độc lập, hai mô hình, hai bộ dữ liệu:

| Bài toán | Dữ liệu | Đầu ra | Số lớp |
|---|---|---|---|
| **Phát hiện lỗi** (Object Detection) | NEU-DET | Loại lỗi, Bounding Box, độ tin cậy | 6 |
| **Phân vùng lỗi** (Semantic Segmentation) | NEU-Seg | Mask theo điểm ảnh | 3 lớp lỗi + nền |

Phân vùng chỉ có `inclusion`, `patches`, `scratches` vì NEU-Seg không có mask cho `crazing`, `pitted_surface`, `rolled-in_scale`. Không dùng mạng đa nhiệm, không suy mask từ box, không dùng kết quả mô hình này để huấn luyện mô hình kia. Tầng ứng dụng có thể sinh mô tả tiếng Việt và trả lời câu hỏi mẫu từ JSON, nhưng chỉ diễn giải kết quả; không thay đổi lõi thị giác.

<a id="define-2"></a>

## 1.2 Đối tượng sử dụng

Nhân viên QC làm việc với ảnh tĩnh, nhận kết quả trên từng ảnh, không cần can thiệp vào mô hình (tham số do nhóm cấu hình sẵn). Hệ thống hỗ trợ quan sát, sàng lọc và lưu vết, không thay thế con người.

<a id="define-3"></a>

## 1.3 Dữ liệu sử dụng

| Nguồn | Quy mô | Nhãn | Vai trò |
|---|---|---|---|
| **NEU-DET** | **[Đã kiểm tra]** 1.800 ảnh 200×200 | 6 lớp, 4.189 Bounding Box (Pascal VOC XML) | Huấn luyện, đánh giá YOLOv8s và baseline detection |
| **NEU-Seg** | **[Đã kiểm tra]** 4.470 ảnh (3.630 training / 840 test) | Mask `uint8` giá trị 0–3 | Huấn luyện, đánh giá U-Net và baseline segmentation |

- **NEU-DET:** 1.440 ảnh train + 360 ảnh validation có sẵn. Nhóm dùng 360 ảnh làm **test**, chia 1.440 ảnh thành 1.152 train / 288 validation nội bộ.
- **NEU-Seg:** chỉ dùng 3.630 ảnh training, tách 90/10 thành 3.267 train / 363 validation nội bộ; giữ 840 ảnh test cho báo cáo cuối. Ánh xạ mask 1–3 sang tên lớp **[Đã kiểm tra trên dữ liệu local]**: số mask chứa giá trị 1/2/3 là 1.460/1.448/1.429, khớp thứ tự `inclusion`/`patches`/`scratches`; còn 182 ảnh single-class trùng NEU-DET có mask khác lớp box, cần audit trước khi huấn luyện.
- **Chồng lấn:** **[Đã kiểm tra]** 900 ảnh trùng nội dung giữa hai bộ, cách chia sẵn không đồng bộ. Vì vậy không gộp hai bộ và không dùng test của bộ này kết luận cho bộ kia.
- **Giới hạn:** cả hai bộ chỉ gồm ảnh **có lỗi** (thép cán nóng), nên không đo được tỉ lệ báo nhầm trên sản phẩm tốt và không suy rộng sang vật liệu khác.
- Giấy phép của cả hai bộ **[Cần xác minh]** trước khi nộp và công bố.

<a id="define-4"></a>

## 1.4 Phương pháp dự kiến

- **Detection:** YOLOv8s pretrained COCO, fine-tune trên NEU-DET (6 lớp).
- **Segmentation:** U-Net encoder–decoder huấn luyện từ đầu trên NEU-Seg, đầu ra 4 kênh. Không dùng checkpoint không rõ nguồn gốc.
- **Baseline (mục 2.5):** **B-D1 — YOLOv8n** (mốc cùng họ, dung lượng nhỏ) cho Detection; **B-S1 — phân ngưỡng Otsu** (mốc không học sâu) cho Segmentation. B-D2/B-S2 chỉ làm nếu đủ tài nguyên. Không dùng zero-shot COCO làm baseline định lượng.
- **Tầng ứng dụng:** JSON từ hai nhánh → LLM/prompt có sẵn sinh mô tả tiếng Việt và trả lời câu hỏi mẫu; không fine-tune LLM/VLM, không sửa box/mask.
- **Tích hợp:** hai mô hình nhận cùng ảnh, tiền xử lý riêng, suy luận độc lập, hiển thị cạnh nhau. Không đặt trước mức mAP/IoU/Dice khi chưa có thực nghiệm.

<a id="define-5"></a>

## 1.5 Phạm vi thực hiện

| Mức | Nội dung |
|---|---|
| **Trong phạm vi bắt buộc** | Detection 6 lớp (NEU-DET) + Segmentation 3 lớp (NEU-Seg); huấn luyện, đánh giá độc lập; so sánh với **B-D1** và **B-S1**; demo web tải một ảnh, hiển thị hai kết quả; phân tích lỗi và giới hạn |
| **Tầng mở rộng ứng dụng** | **Captioning/VQA tiếng Việt** ở giao diện: chuyển JSON kết quả qua LLM/prompt có sẵn (không fine-tune) để sinh mô tả và trả lời câu hỏi mẫu; không sửa box/mask, không fusion |
| **Ngoài phạm vi** | Video, camera giám sát, thời gian thực; fine-tune VLM/LLM lớn từ đầu; mạng đa nhiệm; mask cho 3 lớp không có nhãn; tối ưu triển khai thiết bị biên |

**Mở rộng nghiên cứu không bắt buộc:** Decision Fusion đối chiếu box với mask trên 3 lớp chung, chỉ xem xét sau khi hoàn thành 3 phần trên; không cam kết trong 8 tuần.

<a id="define-6"></a>

## 1.6 Sơ đồ tổng quan hệ thống

<p align="center">
  <img src="assets/define-design/define_system_overview.png" alt="Sơ đồ tổng quan hệ thống" width="88%">
</p>

Người dùng tải ảnh; hệ thống chuyển ảnh đến hai nhánh: YOLOv8s phát hiện và định vị lỗi, U-Net phân vùng lỗi ở cấp điểm ảnh. Hai nhánh trả kết quả độc lập; JSON được chuyển qua tầng ngôn ngữ để sinh mô tả/hỏi đáp trước khi hiển thị. Không có bước hợp nhất box/mask; tầng ngôn ngữ chỉ diễn giải JSON.

<details><summary>Mã Mermaid (có thể chỉnh sửa)</summary>

```mermaid
%%{init: {"htmlLabels": false}}%%
flowchart LR
    U["Người dùng"]:::gray --> A["Tải ảnh<br/>sản phẩm"]:::gray
    A --> B["Hệ thống AI<br/>phân tích ảnh"]:::navy
    B --> D["YOLOv8s<br/>Phát hiện và định vị lỗi"]:::blue
    B --> S["U-Net<br/>Phân vùng khu vực lỗi"]:::teal
    D --> RD["Kết quả Detection<br/>box · lớp · confidence"]:::blue
    S --> RS["Kết quả Segmentation<br/>mask · diện tích"]:::teal
    RD --> T["Sinh mô tả & hỏi đáp<br/>Captioning/VQA"]:::purple
    RS --> T
    T --> R["Hiển thị kết quả<br/>Hình ảnh + Ngôn ngữ"]:::green

    classDef navy fill:#0f2a44,stroke:#0f2a44,color:#ffffff,stroke-width:1px
    classDef blue fill:#dbeafe,stroke:#2563eb,color:#1e3a8a,stroke-width:1px
    classDef teal fill:#ccfbf1,stroke:#0f766e,color:#134e4a,stroke-width:1px
    classDef gray fill:#f1f5f9,stroke:#94a3b8,color:#334155,stroke-width:1px
    classDef green fill:#dcfce7,stroke:#16a34a,color:#14532d,stroke-width:1px
    classDef purple fill:#f3e8ff,stroke:#7e22ce,color:#581c87,stroke-width:1px
```

</details>

<a id="define-7"></a>

## 1.7 Sản phẩm và định hướng đánh giá

**Sản phẩm:** ứng dụng demo hiển thị đồng thời Detection và Segmentation, kèm khung **Captioning/VQA tiếng Việt** sinh từ JSON (LLM/prompt có sẵn, không fine-tune); checkpoint YOLOv8s và U-Net; mã nguồn huấn luyện, suy luận, ứng dụng và cấu hình; dữ liệu mẫu, mô tả dataset và cách chia tập; hướng dẫn cài đặt và tái lập; báo cáo thực nghiệm có **bảng so sánh với baseline** và phân tích lỗi. Module Decision Fusion không bắt buộc.

| Nhiệm vụ | Chỉ số | Tập đánh giá |
|---|---|---|
| Detection | Precision, Recall, mAP@0.5, mAP@0.5:0.95 | 360 ảnh test NEU-DET |
| Segmentation | IoU từng lớp, Dice, mIoU, Pixel Precision, Pixel Recall | 840 ảnh test NEU-Seg |
| Tầng ứng dụng | Độ nhất quán với JSON, độ đọc hiểu, tỉ lệ hallucination, độ trễ | 20–30 ảnh demo + 5–10 câu hỏi mẫu |

- Chỉ báo cáo trên test không tham gia huấn luyện hay chọn checkpoint; ghi rõ tỉ lệ chia và seed.
- Không so sánh trực tiếp mAP với IoU (hai nhiệm vụ khác nhau).
- Captioning/VQA chỉ kiểm tra tính nhất quán với JSON và chất lượng trình bày; không đánh giá bằng mAP/IoU và không gộp vào kết quả thị giác.
- Không đặt ngưỡng tuyệt đối trước; **mục tiêu đo lường là tương đối**: mô hình chính phải được so với baseline cùng nhiệm vụ trên cùng điều kiện, kèm AP/IoU từng lớp, thời gian suy luận và số tham số.

<a id="design"></a>

# 2. Design — Thiết kế hệ thống

Trọng tâm: giữ hai bài toán tách biệt, kiểm soát rò rỉ dữ liệu, và bảo đảm kết quả phản ánh đúng từng mô hình so với baseline.

<a id="design-1"></a>

## 2.1 Nguyên tắc thiết kế

| Nguyên tắc | Nội dung |
|---|---|
| **Hai bài toán tách biệt** | YOLOv8s học 6 lớp (NEU-DET); U-Net phân vùng 3 lớp (NEU-Seg); không dùng chung đầu ra |
| **Cùng ảnh, tiền xử lý riêng** | Hai nhánh nhận cùng ảnh gốc nhưng chuẩn bị đầu vào riêng |
| **Không fusion trong luồng chính** | Hiển thị hai kết quả độc lập |
| **Tầng ngôn ngữ chỉ diễn giải** | Captioning/VQA nhận JSON đã chuẩn hóa; không tạo/sửa box, mask hay lớp |
| **LLM có sẵn, không fine-tune** | Dùng prompt + JSON grounding; có template fallback khi LLM lỗi hoặc không có mạng |
| **Tách dữ liệu đánh giá** | Test không dùng để huấn luyện, chọn epoch, ngưỡng hay checkpoint |
| **Tái lập** | Split, seed lưu thành tệp và băm; cấu hình nằm trong YAML |
| **So sánh công bằng** | Baseline và mô hình chính dùng cùng split, cùng test, cùng bộ đánh giá |

<a id="design-2"></a>

## 2.2 Kiến trúc hệ thống

Gồm ứng dụng web, bước kiểm tra và tiền xử lý ảnh, hai nhánh mô hình độc lập, tầng thu thập kết quả, **tầng sinh ngôn ngữ (Captioning/VQA)** và tầng hiển thị. Hai nhánh không chia sẻ trọng số hay đặc trưng; tầng ngôn ngữ chỉ diễn giải JSON, không can thiệp dự đoán.

<p align="center">
  <img src="assets/define-design/01_system_architecture.png" alt="Sơ đồ kiến trúc hệ thống" width="100%">
</p>

| Thành phần | Vai trò |
|---|---|
| Web Application | Nhận ảnh, hiển thị kết quả; không chứa logic mô hình |
| Upload & Validation | Kiểm tra định dạng, kích thước, khả năng đọc ảnh |
| Preprocessing | Letterbox cho YOLO; resize + chuẩn hóa cho U-Net |
| Detection branch | YOLOv8s + trọng số NEU-DET → box, lớp, confidence |
| Segmentation branch | U-Net + trọng số NEU-Seg → mask, vùng lỗi |
| Result Collection | Chuẩn hóa đầu ra hai nhánh thành JSON; không hợp nhất box/mask |
| Text Generation Layer | Nhận JSON, gọi LLM/prompt có sẵn sinh mô tả tiếng Việt và trả lời câu hỏi mẫu; không sửa box/mask |
| Result Visualization | Vẽ box/mask lên ảnh gốc, hiển thị hai khung hình ảnh và khung ngôn ngữ cạnh nhau |

Hợp đồng dữ liệu giữa các tầng (số liệu chỉ minh họa):

```json
{
  "image_id": "sample_001",
  "detection": {"boxes": [{"x1": 12, "y1": 30, "x2": 88, "y2": 96, "class": "scratches", "confidence": 0.87}]},
  "segmentation": {"mask_path": "results/sample_001_mask.png", "classes_present": ["scratches"], "defect_area_ratio": 0.061},
  "text": {"caption": "Phát hiện 1 vùng lỗi scratches...", "vqa": [{"question": "Lỗi gì?", "answer": "scratches"}]}
}
```

<details><summary>Mã Mermaid (có thể chỉnh sửa)</summary>

```mermaid
%%{init: {"htmlLabels": false}}%%
flowchart LR
    U["Người dùng"]:::gray --> W["Web Application"]:::navy
    W --> V["Image Upload<br/>& Validation"]:::gray
    V --> P["Preprocessing"]:::gray

    subgraph DET["Nhánh Detection — YOLOv8s"]
        direction LR
        YW["YOLOv8s<br/>NEU-DET weights"]:::blue
        YI["YOLOv8s<br/>inference"]:::blue
        YO["Bounding boxes<br/>Defect classes<br/>Confidence scores"]:::blue
        YW -.-> YI
        YI --> YO
    end

    subgraph SEG["Nhánh Segmentation — U-Net"]
        direction LR
        SW["U-Net<br/>NEU-Seg weights"]:::teal
        SI["U-Net<br/>inference"]:::teal
        SO["Segmentation mask<br/>Pixel-level regions"]:::teal
        SW -.-> SI
        SI --> SO
    end

    P --> YI
    P --> SI
    YO --> RC["Result Collection"]:::navy
    SO --> RC
    RC --> TG["Text Generation Layer<br/>Captioning / VQA"]:::purple
    TG --> RV["Result Visualization"]:::green
    RV --> OUT["Display Both Results<br/>Images + Text"]:::green

    classDef navy fill:#0f2a44,stroke:#0f2a44,color:#ffffff,stroke-width:1px
    classDef blue fill:#dbeafe,stroke:#2563eb,color:#1e3a8a,stroke-width:1px
    classDef teal fill:#ccfbf1,stroke:#0f766e,color:#134e4a,stroke-width:1px
    classDef gray fill:#f1f5f9,stroke:#94a3b8,color:#334155,stroke-width:1px
    classDef green fill:#dcfce7,stroke:#16a34a,color:#14532d,stroke-width:1px
    classDef purple fill:#f3e8ff,stroke:#7e22ce,color:#581c87,stroke-width:1px
    style DET fill:#f8fafc,stroke:#bfdbfe,stroke-width:1px
    style SEG fill:#f8fafc,stroke:#99f6e4,stroke-width:1px
```

</details>

<a id="design-3"></a>

## 2.3 Quy trình chuẩn bị dữ liệu

Hai bộ được chuẩn bị theo hai quy trình riêng, không gộp.

<p align="center">
  <img src="assets/define-design/02_data_preparation_pipeline.png" alt="Sơ đồ quy trình chuẩn bị dữ liệu" width="100%">
</p>

**NEU-DET.** **[Đã kiểm tra]** 123 ảnh đa lớp; `crazing_240.xml` nằm sai tập; 174 XML thiếu đuôi `.jpg` trong `filename`; 188 box có tên lớp khác tên thư mục; 3 cặp box trùng. Các lỗi cấu trúc này được **xử lý bằng script chạy ở terminal** (ghép ảnh–XML theo *stem*, đưa `crazing_240.xml` về đúng tập) trước khi khóa split. Giữ nhãn gốc, không tự sửa 188 box lệch thư mục hay xóa box trùng; ghi vào phân tích lỗi. Chia 1.152 / 288 / 360, tỉ lệ 80/20 trên tập train là **[Đề xuất]**; nên chia phân tầng theo lớp.

**NEU-Seg.** **[Đã kiểm tra]** mask giá trị 0–3; 927 ảnh có hơn một lớp lỗi, 17 ảnh đủ ba lớp; 0 mask rỗng; nền chiếm khoảng 87,9% pixel; 4 cặp ảnh trùng (3 ở training, 1 ở test). Chia 3.267 / 363 / 840; khi tách validation nên **gom các ảnh trùng cùng nhóm băm** vào cùng một tập để tránh rò rỉ. Resize ảnh bằng bilinear, mask bằng nearest. Ánh xạ 1–3 có cơ sở từ số mask chứa từng giá trị (1.460/1.448/1.429 khớp `inclusion`/`patches`/`scratches`); vẫn cần kiểm tra trực quan và **audit 182 ảnh** single-class trùng NEU-DET có mask khác lớp box trước khi huấn luyện. Có thêm **11 cặp ảnh test–train tương quan >0,99**, nên gom nhóm băm khi chia validation.

**Chồng lấn 900 ảnh giữa hai bộ:** mỗi mô hình chỉ báo cáo trên test của bộ tương ứng; khi trình bày tích hợp, ghi rõ ảnh minh họa thuộc tập nào và không tuyên bố khả năng tổng quát hóa ngoài dataset. Chỉ **60 ảnh** nằm ở test của cả hai bộ (NEU-DET validation ∩ NEU-Seg test), nên dùng 60 ảnh này làm **demo set held-out**; các ảnh khác chỉ minh họa định tính. Lưu bản đồ ảnh trùng vào `Results/`.

**Tiền xử lý theo nhánh:**

| Nhánh | Việc làm |
|---|---|
| **YOLOv8s** | Chuyển Pascal VOC sang YOLO (`class_id, x_center, y_center, w, h` chuẩn hóa); letterbox 640×640 |
| **U-Net** | Ảnh 256×256 bilinear, mask nearest; chuẩn hóa theo cấu hình huấn luyện |
| **Hiển thị** | Đưa box (scale ngược letterbox) và mask (nearest) về kích thước ảnh gốc |

<details><summary>Mã Mermaid (có thể chỉnh sửa)</summary>

```mermaid
%%{init: {"htmlLabels": false}}%%
flowchart LR
    subgraph DET["Nhánh Detection — NEU-DET"]
        direction LR
        D1["NEU-DET"]:::gray --> D2["Image<br/>inspection"]:::gray
        D2 --> D3["XML annotation<br/>parsing"]:::gray
        D3 --> D4["Bounding box<br/>processing"]:::blue
        D4 --> D5["Dataset<br/>splitting"]:::gray
        D5 --> D6["YOLO format<br/>conversion"]:::blue
        D6 --> D7["Detection<br/>training data"]:::green
    end

    subgraph SEG["Nhánh Segmentation — NEU-Seg"]
        direction LR
        S1["NEU-Seg"]:::gray --> S2["Image / mask<br/>inspection"]:::gray
        S2 --> S3["Mask class<br/>verification"]:::gray
        S3 --> S4["Preprocessing"]:::gray
        S4 --> S5["Dataset<br/>splitting"]:::gray
        S5 --> S6["Segmentation<br/>training data"]:::green
    end

    classDef blue fill:#dbeafe,stroke:#2563eb,color:#1e3a8a,stroke-width:1px
    classDef gray fill:#f1f5f9,stroke:#94a3b8,color:#334155,stroke-width:1px
    classDef green fill:#dcfce7,stroke:#16a34a,color:#14532d,stroke-width:1px
    style DET fill:#f8fafc,stroke:#bfdbfe,stroke-width:1px
    style SEG fill:#f8fafc,stroke:#99f6e4,stroke-width:1px
```

</details>

<a id="design-4"></a>

## 2.4 Thiết kế mô hình chính

**Detection — YOLOv8s** **[Đã chốt]**: pretrained COCO, fine-tune 6 lớp trên NEU-DET, ảnh 640×640 (letterbox), huấn luyện trên Colab/Kaggle hoặc máy có GPU. Đầu ra `(lớp, box, confidence)`. Ngưỡng confidence hiển thị **[Đề xuất]** 0,25, chỉ ảnh hưởng giao diện, không dùng để tính mAP. Kết quả zero-shot COCO chỉ có thể ghi chú định tính về khoảng cách miền; **không dùng làm baseline định lượng** — baseline detection là B-D1 YOLOv8n (mục 2.5).

**Segmentation — U-Net** **[Đã chốt]**: huấn luyện từ đầu trên NEU-Seg; encoder–decoder có skip connection; đầu ra 4 kênh (nền + 3 lớp lỗi); đầu vào 256×256; loss Cross-Entropy + Dice; dự đoán bằng `argmax`. Ưu tiên cấu hình lightweight (16–32 kênh cơ sở) nếu thiếu GPU/RAM. Kết quả chỉ có nghĩa cho 3 lớp có mask.

**Tích hợp hai nhánh** (mức ứng dụng, không ở mức mạng): cùng ảnh gốc, tiền xử lý và checkpoint riêng, đưa về cùng hệ tọa độ ảnh gốc nhưng giữ hai cấu trúc kết quả riêng; giao diện hiển thị hai khung; không có đầu ra hợp nhất, bỏ phiếu hay ghi đè lớp.

**Tầng ứng dụng — Captioning/VQA** **[Đã chốt]**: nhận JSON từ Result Collection; dùng LLM/prompt có sẵn, không fine-tune; prompt chỉ được mô tả lớp, số vùng, vị trí và diện tích có trong JSON. Nếu LLM lỗi, timeout hoặc không có mạng, dùng template tiếng Việt từ JSON. Tầng này không tạo box/mask mới và không phải fusion.

<a id="design-5"></a>

## 2.5 Baseline

Baseline là mốc để trả lời "mô hình chính tốt hơn cái gì, và tốt hơn bao nhiêu". Mỗi baseline phải cùng nhiệm vụ, cùng split, cùng tập test và cùng bộ đánh giá với mô hình chính.

**Tên gọi thống nhất:** **B-D1 = YOLOv8n** và **B-S1 = phân ngưỡng Otsu**. Đây là hai baseline bắt buộc, dùng cùng tên trong phiếu đăng ký, notebook và báo cáo. Zero-shot COCO không phải baseline định lượng.

| Nhiệm vụ | Mã | Baseline | Vai trò | Mức |
|---|---|---|---|---|
| Detection | **B-D1** | **YOLOv8n** (pretrained COCO, fine-tune cùng cấu hình) | Mốc cùng họ, nhỏ hơn: cho biết cỡ `s` có đáng so với `n` | **Bắt buộc** |
| Detection | B-D2 | **Faster R-CNN ResNet50-FPN** (torchvision, pretrained COCO, fine-tune) | Mốc khác họ (hai giai đoạn) | Nên làm nếu đủ GPU |
| Segmentation | **B-S1** | **Phân ngưỡng Otsu**: làm mờ → Otsu (hoặc ngưỡng thích nghi) → lọc thành phần liên thông nhỏ; chỉ phân vùng nhị phân nền–lỗi, không phân lớp | Mốc không học sâu: cho biết học sâu đem lại gì | **Bắt buộc** |
| Segmentation | B-S2 | **U-Net encoder ResNet34 pretrained ImageNet** (cùng loss, split, seed) | So sánh huấn luyện từ đầu và có pretrained | Nên làm nếu đủ GPU |

**Quy tắc so sánh** **[Đề xuất]**
- **Cùng bộ đánh giá:** mọi mô hình detection (kể cả YOLOv8s) được tính mAP bằng **cùng một bộ đánh giá** (ví dụ `torchmetrics` hoặc `pycocotools`) trên cùng dự đoán đã xuất ra, vì mỗi thư viện tính mAP hơi khác nhau.
- **Cùng ngân sách:** baseline dùng cùng split, seed, kích thước ảnh và số epoch tối đa; không tinh chỉnh baseline ít hơn hay nhiều hơn đáng kể so với mô hình chính.
- **Tham số của B-S1** (kích thước làm mờ, ngưỡng, kích thước lọc liên thông) chỉ chọn trên validation nội bộ, không chọn trên test.
- **So sánh nhị phân cho segmentation:** vì B-S1 không phân lớp, so B-S1 với U-Net trên chỉ số **nhị phân nền–lỗi** (gộp lớp 1–3 thành "lỗi", báo IoU và Dice). IoU từng lớp và mIoU chỉ báo cáo cho U-Net và B-S2.
- Kết quả chạy một seed; nếu còn thời gian, chạy 3 seed cho cặp YOLOv8s–YOLOv8n để xem chênh lệch có vượt dao động ngẫu nhiên hay không.

**Bảng kết quả cần có trong báo cáo**

| Detection | mAP@0.5 | mAP@0.5:0.95 | Precision | Recall | Tham số | ms/ảnh |
|---|---|---|---|---|---|---|
| YOLOv8s (chính) | | | | | | |
| B-D1 YOLOv8n | | | | | | |
| B-D2 Faster R-CNN (nếu có) | | | | | | |

| Segmentation | IoU nhị phân | Dice nhị phân | mIoU 3 lớp | Tham số | ms/ảnh |
|---|---|---|---|---|---|
| U-Net from scratch (chính) | | | | | |
| B-S1 Phân ngưỡng cổ điển | | | — | — | |
| B-S2 U-Net pretrained (nếu có) | | | | | |

<a id="design-6"></a>

## 2.6 Quy trình suy luận độc lập

<p align="center">
  <img src="assets/define-design/03_parallel_inference_workflow.png" alt="Sơ đồ quy trình suy luận" width="100%">
</p>

1. **Upload:** chọn một ảnh JPG/PNG. 2. **Validation:** kiểm tra định dạng, kích thước tối thiểu, giải mã, hướng EXIF; tệp lỗi bị từ chối kèm thông báo.  
3. **Preprocessing:** tạo hai đầu vào riêng (letterbox 640×640 cho YOLO; 256×256 chuẩn hóa cho U-Net).  
4. **Suy luận hai nhánh** độc lập (trên một GPU có thể chạy tuần tự; kết quả không phụ thuộc thứ tự): YOLOv8s trả box, lớp, confidence; U-Net trả mask 4 lớp, `argmax`, resize nearest về kích thước gốc.  
5. **Sinh ngôn ngữ:** Result Collection xuất JSON chuẩn hóa; **Text Generation Layer** gọi LLM/prompt có sẵn để sinh caption tiếng Việt và trả lời câu hỏi mẫu; chỉ diễn giải JSON, không sửa box/mask.  
6. **Hiển thị:** vẽ box/mask và hiển thị thêm khung "Báo cáo ngôn ngữ tự nhiên & Hỏi đáp"; không tạo ảnh hợp nhất.

**Xử lý lỗi:** nếu một nhánh thị giác thất bại (thiếu checkpoint, lỗi suy luận), nhánh còn lại vẫn hiển thị và ứng dụng ghi rõ nhánh lỗi. Nếu tầng ngôn ngữ lỗi, timeout hoặc thiếu API key, ứng dụng vẫn hiển thị box/mask và dùng caption template từ JSON; caption ghi rõ nhánh nào thiếu dữ liệu, không suy diễn. Không tự thay thế kết quả giữa hai nhánh.

<details><summary>Mã Mermaid (có thể chỉnh sửa)</summary>

```mermaid
%%{init: {"htmlLabels": false}}%%
flowchart LR
    A["Image<br/>Upload"]:::gray --> B["Image<br/>Validation"]:::gray
    B --> C["Preprocessing"]:::gray
    C --> DI["Detection<br/>Input"]:::blue
    C --> SI["Segmentation<br/>Input"]:::teal

    subgraph DET["Nhánh Detection — YOLOv8s"]
        direction LR
        DI --> Y["YOLOv8s<br/>Inference"]:::blue
        Y --> YB["Bounding<br/>Boxes"]:::blue
        YB --> YC["Defect Classes<br/>& Confidence"]:::blue
    end

    subgraph SEG["Nhánh Segmentation — U-Net"]
        direction LR
        SI --> U["U-Net<br/>Inference"]:::teal
        U --> UM["Segmentation<br/>Mask"]:::teal
        UM --> UR["Defect<br/>Regions"]:::teal
    end

    YC --> RC["Result<br/>Collection"]:::navy
    UR --> RC
    RC --> TG["Text Generation<br/>Layer · Captioning / VQA"]:::purple
    TG --> V["Visualization"]:::green
    V --> O["Display Both Results<br/>Images + Text"]:::green

    classDef navy fill:#0f2a44,stroke:#0f2a44,color:#ffffff,stroke-width:1px
    classDef blue fill:#dbeafe,stroke:#2563eb,color:#1e3a8a,stroke-width:1px
    classDef teal fill:#ccfbf1,stroke:#0f766e,color:#134e4a,stroke-width:1px
    classDef gray fill:#f1f5f9,stroke:#94a3b8,color:#334155,stroke-width:1px
    classDef green fill:#dcfce7,stroke:#16a34a,color:#14532d,stroke-width:1px
    classDef purple fill:#f3e8ff,stroke:#7e22ce,color:#581c87,stroke-width:1px
    style DET fill:#f8fafc,stroke:#bfdbfe,stroke-width:1px
    style SEG fill:#f8fafc,stroke:#99f6e4,stroke-width:1px
```

</details>

<a id="design-7"></a>

## 2.7 Thiết kế huấn luyện

| Mục | YOLOv8s (chính) | U-Net (chính) |
|---|---|---|
| Khởi tạo | COCO pretrained | Từ đầu |
| Kích thước ảnh | 640×640 | 256×256 |
| Epoch | 100, dừng sớm sau 20 epoch | 80, dừng sớm sau 15 epoch |
| Batch | 16 | 8 |
| Tối ưu | SGD, lr 0,01 | AdamW, lr 1e-3, weight decay 1e-4 |
| Loss | Mặc định YOLOv8 | Cross-Entropy + Dice |
| Chọn checkpoint | mAP@0.5 trên validation nội bộ | mIoU trên validation nội bộ |
| Seed | 42 | 42 |

Các giá trị trên là **[Đề xuất]**, điều chỉnh theo GPU và kết quả chạy thử. **Baseline dùng cùng cấu hình**, chỉ khác phần làm nên baseline (YOLOv8n đổi cỡ model; Faster R-CNN và U-Net pretrained theo thư viện tương ứng). Nguyên tắc: chỉ chọn checkpoint trên validation; lưu checkpoint tốt nhất và log theo epoch; ghi phiên bản thư viện, phần cứng, thời gian huấn luyện; nếu thiếu bộ nhớ thì giảm batch hoặc số kênh U-Net trước khi giảm kích thước ảnh và ghi lại trong báo cáo.

**Tầng ngôn ngữ:** không huấn luyện/fine-tune LLM/VLM; chỉ dùng prompt, JSON grounding, template fallback và cache. Nếu dùng API ngoài, ghi rõ nhà cung cấp, phiên bản model và chi phí; không gửi ảnh gốc nếu không cần.

<a id="design-8"></a>

## 2.8 Quy trình đánh giá

<p align="center">
  <img src="assets/define-design/04_model_evaluation_workflow.png" alt="Sơ đồ quy trình đánh giá mô hình" width="100%">
</p>

**Giao thức.** Mỗi mô hình chỉ đánh giá trên tập test tương ứng; cùng split, seed và phiên bản mã cho mọi lần báo cáo; mở test một lần cho kết quả chính (chạy lại phải ghi lý do và không chọn cấu hình theo test); báo kèm số mẫu, phần cứng, thời gian suy luận.

| Nhiệm vụ | Chỉ số | Ghi chú |
|---|---|---|
| Detection | Precision, Recall, mAP@0.5, mAP@0.5:0.95, AP từng lớp | Cùng bộ đánh giá cho YOLOv8s và baseline |
| Segmentation | IoU từng lớp, Dice, mIoU, Pixel Precision/Recall; IoU và Dice nhị phân để so B-S1 | Pixel Precision/Recall tính trên lớp lỗi 1–3, không gộp nền để tránh làm đẹp do mất cân bằng |
| Tầng ứng dụng | Tính nhất quán với JSON (lớp, số vùng, vị trí, diện tích), độ đọc hiểu, tỉ lệ hallucination, độ trễ | 20–30 ảnh demo + 5–10 câu hỏi mẫu; đối chiếu thủ công |

**Phân tích lỗi:** Detection (dương tính giả, âm tính giả, nhầm lớp, bỏ sót lỗi nhỏ, box lệch); Segmentation (sai biên, nhầm lớp, bỏ sót vùng nhỏ, nền bị dự đoán thành lỗi); Tích hợp (hai nhánh độc lập, một nhánh lỗi không chặn nhánh kia, ảnh hợp lệ/sai định dạng/thiếu tệp/quá lớn); **Captioning/VQA** (thêm/bớt lớp, sai số vùng, sai vị trí, bịa thông tin ngoài JSON, lỗi định dạng tiếng Việt). Thời gian suy luận đo riêng từng nhánh và tầng ngôn ngữ sau khởi động, báo trung bình và P95 nếu đủ mẫu. Không so trực tiếp mAP với IoU.

**Đánh giá Captioning/VQA:** không dùng mAP/IoU; chấm từng câu so với JSON, đếm lỗi hallucination, đánh giá độ đọc hiểu và độ trễ trên bộ câu hỏi cố định. Kết quả báo cáo riêng, không gộp vào bảng thị giác.

<details><summary>Mã Mermaid (đã cập nhật có baseline)</summary>

```mermaid
%%{init: {"htmlLabels": false}}%%
flowchart LR
    subgraph DET["Đánh giá Detection"]
        direction LR
        D1["Test images +<br/>ground-truth boxes"]:::gray --> D2["YOLOv8s (chính)"]:::blue
        D1 --> DB["Baseline<br/>B-D1 YOLOv8n · B-D2 Faster R-CNN"]:::note
        D2 --> D3["Cùng bộ đánh giá mAP"]:::blue
        DB --> D3
        D3 --> D4["Precision · Recall<br/>mAP@0.5 · mAP@0.5:0.95"]:::blue
    end

    subgraph SEG["Đánh giá Segmentation"]
        direction LR
        S1["Test images +<br/>ground-truth masks"]:::gray --> S2["U-Net (chính)"]:::teal
        S1 --> SB["Baseline<br/>B-S1 Otsu · B-S2 U-Net pretrained"]:::note
        S2 --> S3["Mask Comparison"]:::teal
        SB --> S3
        S3 --> S4["IoU · Dice · mIoU<br/>IoU/Dice nhị phân"]:::teal
    end

    subgraph APP["Đánh giá tầng ứng dụng"]
        direction LR
        A1["20–30 ảnh demo<br/>5–10 câu hỏi mẫu"]:::gray --> A2["LLM/prompt<br/>Captioning / VQA"]:::purple
        A2 --> A3["Đối chiếu JSON<br/>hallucination · độ trễ"]:::purple
    end

    D4 --> R["Evaluation Report<br/>bảng so với baseline"]:::navy
    S4 --> R
    A3 --> R
    R --> E["Error Analysis"]:::gray
    E --> P["Performance Summary"]:::green
    R -.-> N["Không so trực tiếp<br/>mAP với IoU"]:::note

    classDef navy fill:#0f2a44,stroke:#0f2a44,color:#ffffff,stroke-width:1px
    classDef blue fill:#dbeafe,stroke:#2563eb,color:#1e3a8a,stroke-width:1px
    classDef teal fill:#ccfbf1,stroke:#0f766e,color:#134e4a,stroke-width:1px
    classDef gray fill:#f1f5f9,stroke:#94a3b8,color:#334155,stroke-width:1px
    classDef green fill:#dcfce7,stroke:#16a34a,color:#14532d,stroke-width:1px
    classDef note fill:#ffffff,stroke:#94a3b8,color:#475569,stroke-width:1px,stroke-dasharray:4 3
    classDef purple fill:#f3e8ff,stroke:#7e22ce,color:#581c87,stroke-width:1px
    style DET fill:#f8fafc,stroke:#bfdbfe,stroke-width:1px
    style SEG fill:#f8fafc,stroke:#99f6e4,stroke-width:1px
    style APP fill:#fdf4ff,stroke:#e9d5ff,stroke-width:1px
```

</details>

<a id="design-9"></a>

## 2.9 Thiết kế ứng dụng demo

Streamlit hoặc Gradio, chạy cục bộ **[Đề xuất]**:

| Khu vực | Chức năng |
|---|---|
| Upload | Chọn ảnh JPG/PNG, xem trước, kiểm tra định dạng |
| Detection panel | Ảnh có box, danh sách lớp + confidence, số vùng phát hiện |
| Segmentation panel | Mask chồng lên ảnh gốc, bảng lớp và diện tích vùng lỗi; với 3 lớp không có mask hiển thị rõ "lớp này không hỗ trợ phân vùng" |
| Captioning/VQA panel | Đoạn mô tả tiếng Việt sinh từ JSON; ô hỏi đáp với 5–10 câu hỏi mẫu; ghi rõ nguồn dữ liệu và cảnh báo nếu một nhánh lỗi |
| Controls / Export | Chạy, xóa, chọn ngưỡng hiển thị (không ảnh hưởng đánh giá); bật/tắt LLM; tải ảnh kết quả và JSON gồm detection, segmentation, text |
| Error states | Tệp không hợp lệ, ảnh quá lớn, thiếu checkpoint, một nhánh lỗi |

Không có chế độ Fusion. Hai kết quả thị giác luôn hiển thị riêng với nhãn "Detection" và "Segmentation"; khung **Captioning/VQA** chỉ diễn giải JSON, không tạo box/mask mới. Baseline chỉ dùng cho đánh giá, không đưa vào demo. Khi LLM tắt hoặc lỗi, khung ngôn ngữ dùng template từ JSON.

<a id="design-10"></a>

## 2.10 Cấu trúc dự án và kế hoạch 8 tuần

```text
AI_Industrial_Defect/
├── data/            # NEU-DET (ảnh, XML, split), NEU-Seg (ảnh, mask, split)
├── configs/         # detection.yaml, segmentation.yaml, baseline_*.yaml
├── src/
│   ├── data/        # kiểm kê, chuyển đổi, chia tập
│   ├── detection/   # YOLOv8s, baseline B-D1/B-D2
│   ├── segmentation/# U-Net, baseline B-S1/B-S2
│   ├── evaluation/  # chỉ số, phân tích lỗi
│   ├── application/ # captioning/VQA, prompt, template fallback
│   └── app/         # giao diện demo
├── assets/define-design/
├── Research/
└── Results/         # checkpoint, bảng kết quả, hình
```

| Thời gian | Công việc | Kết quả cần có |
|---|---|---|
| Tuần 1 | Khảo sát bài toán, xác định phạm vi | Phiếu đăng ký, tình huống sử dụng, sơ đồ Define |
| Tuần 2 | Chuẩn bị dữ liệu, thiết kế đánh giá | Dữ liệu đã kiểm tra, split, sơ đồ Design |
| Tuần 3 | **Chạy baseline** (B-D1, B-S1 bắt buộc) | Kết quả mốc cho hai nhiệm vụ |
| Tuần 4–5 | Huấn luyện YOLOv8s và U-Net; B-D2, B-S2 nếu đủ GPU; không huấn luyện LLM | Checkpoint, log huấn luyện |
| Tuần 6 | Tích hợp ứng dụng, tầng Captioning/VQA, xử lý lỗi | Bản demo 3 khung: Detection, Segmentation, Ngôn ngữ |
| Tuần 7 | Đánh giá cùng điều kiện với baseline, phân tích lỗi | Bảng so sánh, phân tích lỗi, thời gian suy luận |
| Tuần 8 | Hoàn thiện báo cáo và bàn giao | Demo, mã nguồn, hướng dẫn, báo cáo |

**Phân công theo gói:** W1 dữ liệu và chuyển đổi · W2 detection (YOLOv8s + baseline) · W3 segmentation (U-Net + baseline) · W4 ứng dụng, tầng ngôn ngữ và kiểm thử tích hợp · W5 báo cáo và tài liệu. Thành viên ghi trong báo cáo nộp, luân phiên kiểm tra chéo.

<a id="design-11"></a>

## 2.11 Quyết định, nội dung cần xác minh và rủi ro

**Đã chốt:** hai mô hình độc lập, không đa nhiệm, không fusion trong luồng chính · YOLOv8s fine-tune COCO; U-Net từ đầu · NEU-DET 1.152/288/360; NEU-Seg 3.267/363/840 · baseline bắt buộc **B-D1 YOLOv8n** và **B-S1 Otsu** · Captioning/VQA ở tầng ứng dụng dùng LLM/prompt có sẵn, không fine-tune · chỉ báo cáo trên test, mục tiêu so với baseline thay vì ngưỡng tuyệt đối.

**Cần xác minh:** giấy phép NEU-DET/NEU-Seg · audit 182 ảnh box–mask bất nhất và 11 cặp gần trùng NEU-Seg · danh sách 900 ảnh trùng (băm nội dung) và 60 ảnh demo both-test · GPU/RAM và thời gian huấn luyện (chạy thử 1–2 epoch) · khóa phiên bản thư viện · phần cứng đo thời gian suy luận.

| Rủi ro | Biện pháp |
|---|---|
| Thiếu GPU/RAM, huấn luyện kéo dài | U-Net lightweight, giảm batch, Colab/Kaggle, lưu checkpoint theo epoch, dừng sớm; baseline nâng cao (B-D2, B-S2) có thể bỏ |
| Nhãn không đồng nhất | Kiểm kê, ghi nhận mẫu lỗi; không tự sửa khi chưa có quy trình |
| Xung đột thư viện, hai nhánh khác tiền xử lý | Khóa phiên bản, môi trường ảo, cấu hình riêng, kiểm thử từng nhánh trước khi ghép |
| Một nhánh lỗi làm hỏng ứng dụng | Bọc ngoại lệ từng nhánh, hiển thị trạng thái lỗi riêng |
| LLM bịa thông tin hoặc sai số liệu | Prompt chỉ dùng JSON; kiểm tra tự động các trường bắt buộc; template fallback; đánh giá hallucination trên bộ mẫu |
| LLM chậm, tốn chi phí hoặc không có mạng | Cache, timeout, model nhẹ/local, nút tắt; không chặn kết quả thị giác |
| Lộ dữ liệu khi gọi API ngoài | Chỉ gửi JSON đã lược thông tin; không gửi ảnh gốc; ghi rõ nhà cung cấp |
| Chỉ có ảnh có lỗi, một lớp mask thiếu | Ghi rõ không đo được báo nhầm trên sản phẩm tốt; demo nêu rõ lớp không hỗ trợ phân vùng |
| Baseline khác thư viện làm lệch mAP | Dùng chung một bộ đánh giá trên dự đoán đã xuất |

<a id="design-12"></a>

## 2.12 Hướng mở rộng: Decision Fusion (không bắt buộc)

Chỉ xem xét sau khi hai mô hình đã huấn luyện, đánh giá và tích hợp ổn định. Mục tiêu: kiểm tra việc đối chiếu box của YOLOv8s với mask của U-Net có cải thiện độ tin cậy hay không.

- Ghép mỗi box với vùng mask theo IoU hoặc tỉ lệ diện tích mask nằm trong box; chỉ xử lý 3 lớp chung `inclusion`, `patches`, `scratches`.
- Chốt ngưỡng ghép trước khi đánh giá, không chỉnh theo test; so sánh với từng mô hình riêng trên cùng tập ảnh.
- Báo cáo coverage, độ nhất quán lớp, tỉ lệ ghi đè đúng và sai.
- Nếu không đủ thời gian hoặc không tốt hơn, báo cáo trung thực và không đưa vào sản phẩm bắt buộc. Không fine-tune VLM/LLM lớn; Captioning/VQA đã nằm ở tầng ứng dụng, không phải Decision Fusion.